# IMC2025: integrated inference and evaluation
Uses `imc.pipeline`: DINOv2 retrieval -> SuperPoint / LightGlue -> COLMAP geometric verification and SfM -> validated CSV.
These are pretrained models: this notebook does not train network weights. Start with one train dataset; see README for setup and offline Kaggle use.


In [ ]:
from pathlib import Path
import sys
import torch
from imc.data import DATA_ROOT
from imc.pipeline import Config, resolve_device, run_pipeline
from imc.evaluate import score_submission

ROOT = DATA_ROOT
SPLIT = 'train'  # change to 'test' for inference without labels
DATASETS = ['ETs']  # None processes every dataset; use None for final test inference
OUTPUT = Path('outputs/train_submission.csv')
CONFIG = Config(k=20, max_keypoints=512, resize=512, min_inliers=30)

DEVICE = resolve_device()  # auto: XPU, CUDA, CPU; or specify "xpu" / "cuda"
print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "device:", DEVICE)
if DEVICE.type in ("xpu", "cuda"):
    print("GPU:", getattr(torch, DEVICE.type).get_device_name(DEVICE))
else:
    print("Using CPU; select your GPU Python kernel if this is unexpected.")


In [ ]:
submission = run_pipeline(
    root=ROOT, split=SPLIT, datasets=DATASETS, output=OUTPUT,
    config=CONFIG, work_dir='outputs/colmap',
    device=DEVICE,
    # dino_repo='/path/to/local/dinov2',
    # dino_weights='/path/to/dinov2_vits14_pretrain.pth',
)
print(submission)


In [ ]:
if SPLIT == 'train':
    scores = score_submission(submission, ROOT / 'train_labels.csv', ROOT / 'train_thresholds.csv')
    scores.to_csv(OUTPUT.with_suffix('.scores.csv'))
    display(scores)


## Experiments
Use `evaluation.ipynb` to inspect shortlist coverage first. Compare k=10/20/30, CLS/MAC descriptors, and 512 vs 2048 keypoints on fixed validation datasets, recording score, registered images and runtime. Keep other datasets held out for final evaluation. Each run saves `.run.json` and a separate COLMAP work directory.
For Kaggle, use SPLIT='test', DATASETS=None, OUTPUT=Path('/kaggle/working/submission.csv'), and attach the repo, dependencies and cached pretrained weights before disabling internet.
